# 04 — Unigram and Bigram Naive Bayes Models
Implements the report's 80/20 split, `random_state=42`, TF-IDF and Multinomial Naive Bayes approach.

In [ ]:
!pip -q install nltk wordcloud seaborn scikit-learn
import re, numpy as np, pandas as pd, nltk, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
for r in ["punkt","punkt_tab","stopwords","wordnet","omw-1.4"]: nltk.download(r, quiet=True)

def clean_tweet(tweet):
    if pd.isna(tweet): return ""
    tweet=str(tweet)
    tweet=re.sub(r"https?://\S+|www\.\S+", " url ", tweet, flags=re.I)
    tweet=re.sub(r"@\w+", " user ", tweet)
    tweet=re.sub(r"#(\w+)", r"\1", tweet)
    tweet=re.sub(r"[^\w\s]", " ", tweet)
    return re.sub(r"\s+", " ", tweet).strip().lower()

def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=True, custom_stopwords=None):
    tokens=word_tokenize(clean_tweet(text))
    stops=set(stopwords.words("english")) if remove_stopwords else set()
    stops.update(custom_stopwords or [])
    tokens=[w for w in tokens if w not in stops and w.strip()]
    if stem:
        stemmer=PorterStemmer(); tokens=[stemmer.stem(w) for w in tokens]
    if lemmatize:
        lemma=WordNetLemmatizer(); tokens=[lemma.lemmatize(w) for w in tokens]
    return " ".join(tokens)

def load_csv():
    uploaded=files.upload()
    name=next((n for n in uploaded if n.lower().endswith(".csv")),None)
    if not name: raise ValueError("Upload a CSV file.")
    data=pd.read_csv(name)
    if not {"tweet_text","label"}.issubset(data.columns):
        raise ValueError("CSV must contain columns named tweet_text and label. Rename your columns if needed.")
    data=data.dropna(subset=["label"]).copy()
    data["tweet_text"]=data["tweet_text"].fillna("").astype(str)
    return data

In [ ]:
df=load_csv()
df["tweet_text"]=df["tweet_text"].apply(clean_tweet)
df=df[df["tweet_text"].str.strip().ne("")].copy()
X,y=df["tweet_text"],df["label"].astype(str)
if y.nunique()<2: raise ValueError("At least two labels are needed.")
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,
    stratify=y if y.value_counts().min()>=2 else None)
print("Train:",len(X_train),"Test:",len(X_test))

## Train and evaluate unigram and bigram models

In [ ]:
def run_model(name,ngram_range):
    vec=TfidfVectorizer(ngram_range=ngram_range,max_features=10000)
    A=vec.fit_transform(X_train); B=vec.transform(X_test)
    model=MultinomialNB().fit(A,y_train); pred=model.predict(B)
    print(f"\n{name} accuracy: {accuracy_score(y_test,pred):.4f}")
    print(classification_report(y_test,pred,zero_division=0))
    labels=sorted(y_test.unique()); cm=confusion_matrix(y_test,pred,labels=labels)
    plt.figure(figsize=(max(6,len(labels)),max(5,len(labels)*0.7)))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels)
    plt.title(f"Confusion Matrix — {name}"); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()
    return {"model":name,"accuracy":accuracy_score(y_test,pred)}
results=[run_model("Unigram TF-IDF",(1,1)),run_model("Bigram TF-IDF",(2,2))]
comparison=pd.DataFrame(results); display(comparison)
comparison.to_csv("unigram_bigram_model_comparison.csv",index=False)